In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    roc_auc_score
)

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

# 1. Configuration du serveur d'expérience MLflow
mlflow.set_tracking_uri("http://localhost:5000")  # Ou dossier local 'file:./mlruns'
mlflow.set_experiment("Diabetes_Risk_Classification")

# 2. Chargement et Préparation des Données
df = pd.read_csv('../data/processed/df_classified_target.csv')
df.columns = df.columns.str.strip()

cols_to_drop = [col for col in ['Cluster', 'risk_category'] if col in df.columns]
X = df.drop(columns=cols_to_drop)
y = df['risk_category']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Définition des Modèles et Grilles d'Hyperparamètres
models_config = {
    'LogisticRegression': {
        'pipeline': ImbPipeline([
            ('scaler', StandardScaler()),
            ('smote', SMOTE(random_state=42)),
            ('model', LogisticRegression(random_state=42))
        ]),
        'params': {'model__C': [0.1, 1.0, 10.0]}
    },
    'RandomForest': {
        'pipeline': ImbPipeline([
            ('scaler', StandardScaler()),
            ('smote', SMOTE(random_state=42)),
            ('model', RandomForestClassifier(random_state=42))
        ]),
        'params': {
            'model__n_estimators': [50, 100],
            'model__max_depth': [3, 5, 10]
        }
    },
    'XGBoost': {
        'pipeline': ImbPipeline([
            ('scaler', StandardScaler()),
            ('smote', SMOTE(random_state=42)),
            ('model', xgb.XGBClassifier(random_state=42, eval_metric='logloss'))
        ]),
        'params': {
            'model__n_estimators': [50, 100],
            'model__max_depth': [3, 5],
            'model__learning_rate': [0.01, 0.1]
        }
    }
}

# 4. Boucle d'Exécution & Logging MLflow
best_global_f1 = 0
best_model_run_id = None
best_model_name = ""

for model_name, config in models_config.items():
    with mlflow.start_run(run_name=f"Run_{model_name}"):
        
        # Validation Croisée & Optimisation
        grid = GridSearchCV(config['pipeline'], config['params'], cv=5, scoring='f1', n_jobs=-1)
        grid.fit(X_train, y_train)
        
        best_pipeline = grid.best_estimator_
        y_pred = best_pipeline.predict(X_test)
        y_proba = best_pipeline.predict_proba(X_test)[:, 1] if hasattr(best_pipeline, "predict_proba") else None
        
        # Calcul des Métriques de Performance
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        roc_auc = roc_auc_score(y_test, y_proba) if y_proba is not None else 0.0
        
        # A. Logging des Paramètres dans MLflow
        mlflow.log_params(grid.best_params_)
        mlflow.log_param("test_split_ratio", 0.2)
        mlflow.log_param("resampling_method", "SMOTE")
        
        # B. Logging des Métriques
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("precision", prec)
        mlflow.log_metric("recall", rec)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("roc_auc", roc_auc)
        
        # C. Enregistrement de la Matrice de Confusion comme Artefact
        plt.figure(figsize=(5, 4))
        sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues')
        plt.title(f'Matrice de Confusion - {model_name}')
        plt.xlabel('Prédit')
        plt.ylabel('Réel')
        
        cm_path = f"confusion_matrix_{model_name}.png"
        plt.savefig(cm_path)
        plt.close()
        
        mlflow.log_artifact(cm_path)
        if os.path.exists(cm_path):
            os.remove(cm_path)
            
        # D. Logging du Modèle / Pipeline
        signature = infer_signature(X_train, y_pred)
        mlflow.sklearn.log_model(
            sk_model=best_pipeline,
            artifact_path="model",
            signature=signature
        )
        
        print(f"✅ [{model_name}] F1-Score: {f1:.4f} | Accuracy: {acc:.4f}")
        
        # Suivi du meilleur modèle global
        if f1 > best_global_f1:
            best_global_f1 = f1
            best_model_name = model_name
            best_model_run_id = mlflow.active_run().info.run_id

# 5. Enregistrement du Meilleur Modèle dans le Model Registry
print(f"\n🏆 Champion Model : {best_model_name} (Run ID: {best_model_run_id})")

model_uri = f"runs:/{best_model_run_id}/model"
registered_model = mlflow.register_model(
    model_uri=model_uri,
    name="Diabetes_Risk_Classifier"
)

print(f"✅ Modèle enregistré dans le Model Registry MLflow sous le nom 'Diabetes_Risk_Classifier' (Version 1).")

In [4]:
!pip install -U email-validator mlflow